# Curso de MNE-Python aplicado a TUSZ
## Módulo 0 · Bienvenida y preparación del entorno

Este curso te lleva de cero a tener un **dataset listo para redes neuronales** a partir
de los EDF de TUSZ, usando [MNE-Python](https://mne.tools), la librería de referencia
para EEG/MEG en Python. Todo se hace con **tus propios ficheros** de `D:\Dani\TFM IA\00.TUSZ\v2.0.6`
y el ejemplo principal es una **crisis de ausencia** (`absz`).

| Módulo | Qué aprendes | Fase del TFM |
|---|---|---|
| 00 | Instalación, conceptos y mapa de objetos de MNE | — |
| 01 | `Raw` e `Info`: leer un EDF, canales, datos, unidades | Fase 1 |
| 02 | Visualizar: señal, espectro, posiciones de electrodos | Fase 1 |
| 03 | Anotaciones: convertir `.csv`/`.csv_bi` de TUSZ en `mne.Annotations` | Fase 1 |
| 04 | Referencias y montajes: construir el montaje **TCP** de 22 canales | Fase 1-2 |
| 05 | Filtrado, *notch* y remuestreo | Fase 2 |
| 06 | Artefactos e ICA (y cuándo **no** usarla) | Fase 2 |
| 07 | `Epochs`: ventanas etiquetadas ictal / preictal / interictal | Fase 2 |
| 08 | Espectro y tiempo-frecuencia: bandas, espectrogramas, Morlet | Fase 2-5 |
| 09 | De MNE a ML: `X`, `y`, `groups`, baseline con scikit-learn y exportación a PyTorch | Fase 3 |

Cada módulo tiene teoría corta, código que funciona con tus datos y **🧪 ejercicios**
con la solución justo debajo (intenta hacerlos antes de mirarla).

## 0.1 Instalación

En la terminal (o Anaconda Prompt) del entorno que uses para el TFM:

```bash
pip install -U mne mne-qt-browser matplotlib pandas scipy scikit-learn jupyter
```

- `mne` — la librería (este curso está probado con **MNE 1.13**; funciona desde la 1.8).
- `mne-qt-browser` — visor interactivo rápido para `raw.plot()` (opcional pero muy recomendable).

Los notebooks deben estar en la misma carpeta que `tusz_utils.py`
(`D:\Dani\TFM IA\03.Codigo\Curso Claude MNE`), porque a partir del módulo 05 lo importamos.

In [1]:
import mne, numpy, scipy, matplotlib, pandas, sklearn
for m in (mne, numpy, scipy, matplotlib, pandas, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")

mne          1.13.2
numpy        2.5.1
scipy        1.18.0
matplotlib   3.11.1
pandas       3.0.5
sklearn      1.9.0


`mne.sys_info()` imprime un diagnóstico completo (útil si algo falla y tienes que pedir ayuda):

In [2]:
mne.sys_info()

Platform             Windows-11-10.0.26200-SP0
Python               3.13.9 (tags/v3.13.9:8183fa5, Oct 14 2025, 14:09:13) [MSC v.1944 64 bit (AMD64)]
Executable           C:\Users\Usuario\PycharmProjects\M4_DL\.venv\Scripts\python.exe
CPU                  AMD Ryzen 7 3700X 8-Core Processor (16 cores)
Memory               15.9 GiB

Core
 + mne               1.13.2 (latest release)
 + numpy             2.5.1 (OpenBLAS 0.3.33.112.0 with 16 threads via pthreads)
 + scipy             1.18.0
 + matplotlib        3.11.1 (backend=module://matplotlib_inline.backend_inline)

Numerical (optional)
 + scikit-learn      1.9.0
 + threadpoolctl     3.6.0
 + pandas            3.0.5
 + h5py              3.14.0
 - unavailable       numba, nibabel, nilearn, dipy, openmeeg, python-picard, jamica, cupy, h5io

Visualization (optional)
 - unavailable       pyvista, pyvistaqt, vtk, qtpy, ipympl, pyqtgraph, mne-qt-browser, ipywidgets, trame, trame_client, trame_server, trame_pyvista, trame_vtk, trame_vuetify

Ec

## 0.2 Rutas

Todas las rutas del curso salen de aquí. Si mueves los datos, solo cambias estas dos líneas
(o defines las variables de entorno `TUSZ_ROOT` / `TUSZ_INVENTORY`).

In [3]:
# Las rutas se definen en un único sitio: tusz_utils.py
from tusz_utils import TUSZ_ROOT, INVENTORY, SALIDAS
print(TUSZ_ROOT, TUSZ_ROOT.exists())
print(INVENTORY, INVENTORY.exists())
print(SALIDAS)

D:\Dani\TFM IA\00.TUSZ\v2.0.6 True
D:\Dani\TFM IA\02.Notas\tusz_inventory_out True
D:\Dani\TFM IA\04_datos_derivados\curso_MNE


## 0.3 El mapa mental de MNE

Casi todo en MNE gira alrededor de **cinco objetos**. Si entiendes esto, entiendes la librería:

```
  EDF en disco
      │  mne.io.read_raw_edf()
      ▼
  ┌──────────┐   .info  →  Info  (metadatos: canales, sfreq, tipos, posiciones…)
  │   Raw    │   .annotations → Annotations (onset, duration, description)
  │ continuo │
  └──────────┘
      │  mne.Epochs(raw, events) / mne.make_fixed_length_epochs()
      ▼
  ┌──────────┐   array (n_epochs, n_canales, n_muestras)  ← ¡esto es la X de tu red!
  │  Epochs  │   .metadata → DataFrame con etiquetas (ictal/interictal…)
  └──────────┘
      │  .average()                    │ .compute_psd() / .compute_tfr()
      ▼                                ▼
  ┌──────────┐                  ┌───────────────────────┐
  │  Evoked  │                  │ Spectrum / TFR        │
  └──────────┘                  └───────────────────────┘
```

Tres reglas que ahorran muchos disgustos:

1. **Las unidades internas son SI**: el EEG está en **voltios**, no en µV. Multiplica por `1e6` para µV.
2. **Casi todos los métodos modifican el objeto *in place*** (`raw.filter()`, `raw.crop()`,
   `raw.pick()`…) y además lo devuelven. Si quieres conservar el original: `raw.copy().filter(...)`.
3. **Los tiempos van en segundos** y las muestras en enteros: `muestra = segundos * sfreq`.

## 0.4 Primer contacto

Para ver que todo funciona, abrimos el fichero de ejemplo del curso: paciente `aaaaabqv`,
sesión `s001`, con **3 crisis de ausencia** anotadas.

In [4]:
EDF = TUSZ_ROOT / "edf/train/aaaaabqv/s001_2004/02_tcp_le/aaaaabqv_s001_t000.edf"
raw = mne.io.read_raw_edf(EDF, preload=False)
raw

Extracting EDF parameters from D:\Dani\TFM IA\00.TUSZ\v2.0.6\edf\train\aaaaabqv\s001_2004\02_tcp_le\aaaaabqv_s001_t000.edf...
Setting channel info structure...
Creating raw.info structure...


<RawEDF | aaaaabqv_s001_t000.edf, 33 x 359250 (1437.0 s), ~29 KiB, data not loaded>

Si ves una tabla con 33 canales, 250 Hz y ~24 minutos de duración: **estás listo**. ✅

### Consejo de verbosidad
MNE imprime mucho. Puedes bajarlo globalmente con `mne.set_log_level("WARNING")`
o por llamada con `verbose="error"`. En el curso lo dejamos en `WARNING` a partir de aquí.

In [5]:
mne.set_log_level("WARNING")